In [ ]:
#SFTP 

# # Use
# cnopts = pysftp.CnOpts()
# cnopts.hostkeys = None


# #Get Data from PS Data Export Manager SFTP folder, and move to BigQuery
# with pysftp.Connection(
#     host="sftp.illuminateed.com",
#     username="greendottn",
#     password="*********",
#     cnopts=cnopts
# ) as sftp:
#     # Download a remote file to the local machine
#     remote_file = "/greendottn/custom_report_standards_2024"
#     local_file = "local_file.csv"
#     sftp.get(remote_file, local_file)



In [30]:
#blob name is the name of the file once uploaded
#file_path is local file path
#bucket name is GC bucket unique bucket name

%load_ext autoreload
%autoreload 2

from modules.buckets import create_bucket, upload_to_bucket
import os

# Set the GOOGLE_APPLICATION_CREDENTIALS environment variable in order to interact
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = 'greendotdataflow-848329b50f47.json'


#Create the bucket, and upload to that bucket. If already created, bypass
create_bucket('psholdingbucket5')

#Upload file to bucket, demonstrates if overwritten or newfile. 
#End File Name,  Local File Path, Bucket Name
upload_to_bucket('students.csv' , 'students.csv', 'psholdingbucket5' )

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Bucket psholdingbucket5 already exists.
File students.csv uploaded to psholdingbucket5 in students.csv.
File students.csv was overwritten in psholdingbucket5.


In [ ]:
from google.cloud import bigquery



# Set your Google Cloud project ID
project_id = 'greendotdataflow'

# Create a BigQuery client
client = bigquery.Client(project=project_id)

# Specify the dataset and table name
dataset_id = 'students'
table_id = 'students_table'

# Specify the Cloud Storage URI for your CSV file
csv_uri = 'gs://psholdingbucket3/students.csv'

# Construct the dataset and table references
dataset_ref = client.dataset(dataset_id)
table_ref = dataset_ref.table(table_id)

# Configure the job configuration
job_config = bigquery.LoadJobConfig(
    schema=[
        bigquery.SchemaField("column1", "STRING"),
        bigquery.SchemaField("column2", "STRING"),
        bigquery.SchemaField("column3", "STRING"),
        bigquery.SchemaField("column4", "STRING"),
        bigquery.SchemaField("column5", "STRING"),

    ],
    skip_leading_rows=0,  # Skip the header row in the CSV file
    source_format=bigquery.SourceFormat.CSV,
)

# Load data into the table from the CSV file
job = client.load_table_from_uri(csv_uri, table_ref, job_config=job_config)
job.result()  # Wait for the job to complete

print(f'Table {dataset_id}.{table_id} created from CSV file: {csv_uri}')


In [ ]:
#Create the table based on the csv upload